In [ ]:


import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import sys
sys.path.append(str(Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen' / 'config'))
import plot as pt
import helpers


PATH_PLOTS = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('EXPORT Location: ' + str(PATH_PLOTS))
pd.options.display.float_format = '{:.2f}'.format


EXPORT=False



***

Road_3

***

In [ ]:


indicator = 'Road_3'
plot_name = 'pavement_conditions'

df_counties = pd.read_excel(PATH_PLOTS/'Data'/f"{indicator} Pavement Conditions and Backlog.xlsx", sheet_name='PCI by County', skiprows=3)
df_counties = df_counties.rename(columns={'Unnamed: 1':'County'})
df_counties = df_counties.drop('Unnamed: 0', axis=1)

df_mpo = df_counties[df_counties['County']=='SACOG Region']
df_mpo['County'] = 'SACOG (Average)'
df_mpo = pd.melt(df_mpo, id_vars = ['County'], var_name = 'Year', value_name = 'PCI')

df_plot = df_counties[df_counties['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
df_plot = pd.melt(df_plot, id_vars = ['County'], var_name = 'Year', value_name = 'PCI')

display(df_plot.head(), df_mpo.head())


fig = px.line(df_plot, x='Year', y='PCI', color='County', color_discrete_map=pt.colors_counties, markers=True)
title = '<b>Local Street and Road Pavement Condition, by County</b>  <br><sup>Six-County Sacramento Region</sup> '
# fig.update_yaxes(dtick=5, range = [49,81])
fig.update_traces(hovertemplate="%{y}")

fig.add_trace(go.Scatter(x=df_mpo["Year"], y=df_mpo['PCI'], name = 'SACOG (Average)', line=go.scatter.Line(color="#2C3539", dash="dot")))


pt.plot_agol(fig, EXPORT, title, indicator, plot_name)


***

Road_4

***

In [ ]:

# indicator = 'Road_4'
# plot_name = 'EV_sales'
# end_year=2023

# df_counties = pd.read_excel(PATH_PLOTS/'Data'/f"{indicator} ZEV_Sales download.xlsx", sheet_name='County')

# df_plot = df_counties.copy()
# df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
# df_plot = df_plot[df_plot['Data_Year'] == end_year]

# df_plot = df_plot.groupby(['Data_Year', 'County'], as_index = False)['Number of Vehicles'].sum()
# df_mpo  = df_plot.groupby(['Data_Year'          ], as_index = False)['Number of Vehicles'].sum()
# df_mpo['County'] = 'Total'

# df_plot = pd.concat([df_plot, df_mpo])

# color_map = {
#     'El Dorado':'#1E90FF'
#    , 'Placer':'#1E90FF'
#    , 'Sacramento':'#1E90FF'
#    , 'Sutter':'#1E90FF'
#    , 'Yolo':'#1E90FF'
#    , 'Yuba':'#1E90FF'
#     , 'Total':'#9DC209'
# }


# fig = px.bar(df_plot, x='County', y='Number of Vehicles'
#             , color='County'
#             , color_discrete_map=color_map)


# title = '<b>Zero Emission Vehicle Sales, 2023</b>   <br><sup>6-County Sacramento Region</sup> '
# fig.update_yaxes(tick0=0, dtick=1000, range = [0, 5100], tickformat = ',.0f')
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(showlegend=False)

# EXPORT=False
# pt.plot_agol(fig, EXPORT, title, indicator, plot_name)
# EXPORT=False


In [ ]:


indicator = 'Road_4'
plot_name = 'EV_sales_per_1k_pop'
end_year = 2023


df_counties = pd.read_excel(PATH_PLOTS/'Data'/f"{indicator} ZEV_Sales download.xlsx", sheet_name='County')

df_plot = df_counties.copy()
df_plot = df_plot[df_plot['Data_Year'].isin(helpers.sequence(2012, end_year, 1))]
df_plot = df_plot.rename(columns = {'Data_Year':'Year'})

df_state  = df_plot.groupby(['Year'], as_index = False)['Number of Vehicles'].sum()
df_state['Geography'] = 'California'

df_mpo = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]

df_mpo  = df_mpo.groupby(['Year'], as_index = False)['Number of Vehicles'].sum()
df_mpo['Geography'] = 'SACOG'

df_pop = pd.read_excel(PATH_PLOTS/'Data'/'Pop_5 DOF MPO.xlsx', sheet_name='Data')
df_pop = df_pop[df_pop['Year'].isin(helpers.sequence(2012, end_year, 1))]
df_pop = df_pop[['MPO', 'Year', 'Population']]
df_pop = df_pop.rename(columns = {'MPO':'Geography'})

df_mpo = df_mpo.merge(df_pop[df_pop['Geography'] == 'SACOG'], on = ['Year', 'Geography'])

df_pop = df_pop.groupby(['Year'], as_index = False)['Population'].sum()
df_pop['Geography'] = 'California'

df_state = df_state.merge(df_pop, on = ['Year', 'Geography'])
df_plot = pd.concat([df_mpo, df_state])

df_plot['Sales/1k'] = round(df_plot['Number of Vehicles']/df_plot['Population']*1000, 1)

display(df_plot.head())


fig = px.bar(df_plot, y='Sales/1k', x='Year', color='Geography', barmode='group', color_discrete_map=pt.colors_peers)
title = '<b>Zero Emission Sales per 1,000 People, 2023</b><br><sup>Six-County Sacramento Region</sup> '
# fig.update_yaxes(tick0=0, dtick=2, range = [0, 12.1])
fig.update_xaxes(tick0=0, dtick=1, range = [df_plot['Year'].min()-0.5, df_plot['Year'].max()+0.5])
fig.update_traces(hovertemplate='%{y}')

pt.plot_agol(fig, EXPORT, title, indicator, plot_name)


In [ ]:


indicator = 'Road_4'
plot_name = 'EV_sales'
end_year=2023

df_counties = pd.read_excel(PATH_PLOTS/'Data'/f"{indicator} ZEV_Sales download.xlsx", sheet_name='County')

df_plot = df_counties.copy()
df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
df_plot = df_plot.groupby(['Data_Year', 'County'], as_index = False)['Number of Vehicles'].sum()
df_plot = df_plot[df_plot['Data_Year'].isin(helpers.sequence(2010, end_year, 1))]


df_plot['Sort'] = pd.Categorical(df_plot['County'], [
    'Sacramento'
    , 'Placer'
    , 'Yolo'
    , 'El Dorado'
    , 'Sutter'
    , 'Yuba'
])

df_plot = df_plot.sort_values(['Sort', 'Data_Year'], ascending = [True, True]).drop(['Sort'], axis = 1).reset_index(drop = True)

display(df_plot.head())

fig = px.bar(df_plot, x='Data_Year', y='Number of Vehicles', color='County', color_discrete_map=pt.colors_counties)
title = '<b>Zero Emission Vehicle Sales, 2023</b>   <br><sup>Six-County Sacramento Region</sup> '
fig.update_yaxes(tickformat = ',.0f')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})


pt.plot_agol(fig, EXPORT, title, indicator, plot_name)
